In [34]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
import os

# MultiQA embeddings
q_emb = np.load("/kaggle/input/multi-qa-distilbert-dot-v1/st_qsts_embeddings_train-multi-qa-distilbert-dot-v1.npy")  # shape: (N, D)
d_emb = np.load("/kaggle/input/multi-qa-distilbert-dot-v1/st_docs_embeddings_train-multi-qa-distilbert-dot-v1.npy")  # shape: (N, D)

In [35]:
# --- Information for q_emb ---
print("- Questions Embeddings:")

print(f"Array Shape: {q_emb.shape}")
print(f"Data Type: {q_emb.dtype}")
print(f"Number of Dimensions: {q_emb.ndim}")
print(f"Total Number of Elements: {q_emb.size}")


print("\n" + "="*30 + "\n") # Separator

# --- Information for d_emb ---
print("- Documents Embeddings:")
print(f"Array Shape: {d_emb.shape}")
print(f"Data Type: {d_emb.dtype}")
print(f"Number of Dimensions: {d_emb.ndim}")
print(f"Total Number of Elements: {d_emb.size}")


- Questions Embeddings:
Array Shape: (32433, 768)
Data Type: float32
Number of Dimensions: 2
Total Number of Elements: 24908544


- Documents Embeddings:
Array Shape: (324330, 768)
Data Type: float32
Number of Dimensions: 2
Total Number of Elements: 249085440


In [36]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

embedding_scores = []
documents_per_question = 10 # Assuming each question has a fixed number of associated documents

# Iterate through each question embedding
for i in range(q_emb.shape[0]):
    # Get the current question embedding
    q_emb_i = q_emb[i].reshape(1, -1)
    
    start_index = i * documents_per_question
    end_index = (i + 1) * documents_per_question

    # Get the embeddings of the associated documents
    related_document_embeddings = d_emb[start_index:end_index]
    # Calculate cosine similarity
    cosine_scores = cosine_similarity(q_emb_i, related_document_embeddings)[0].tolist()
    
    # Calculate dot product (which is equivalent to linear_kernel for unnormalized vectors
    dot_product_scores = linear_kernel(q_emb_i, related_document_embeddings)[0].tolist()
    
    embedding_scores.append({
        'question_index': i,
        'associated_document_indices': list(range(start_index, end_index)),
        'cosine_similarity_scores': cosine_scores,
        'dot_product_scores': dot_product_scores
    })

In [40]:
for i, result in enumerate(embedding_scores[:3]):
    print(f"\n--- Results for Question {i} ---")
    print(f"  Question Index: {result['question_index']}")
    print(f"  Associated Documents (Indices): {result['associated_document_indices']}")

    print("\n  Similarity Scores:")
    for j, doc_index in enumerate(result['associated_document_indices']):
        print(f"    Document {j+1} (Index: {doc_index}):")
        print(f"      Cosine Similarity: {result['cosine_similarity_scores'][j]:.4f}")
        print(f"      Dot Product: {result['dot_product_scores'][j]:.4f}")


--- Results for Question 0 ---
  Question Index: 0
  Associated Documents (Indices): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

  Similarity Scores:
    Document 1 (Index: 0):
      Cosine Similarity: 0.2384
      Dot Product: 18.9563
    Document 2 (Index: 1):
      Cosine Similarity: 0.3183
      Dot Product: 27.1305
    Document 3 (Index: 2):
      Cosine Similarity: 0.3593
      Dot Product: 29.1697
    Document 4 (Index: 3):
      Cosine Similarity: 0.2471
      Dot Product: 20.6376
    Document 5 (Index: 4):
      Cosine Similarity: 0.2508
      Dot Product: 19.4539
    Document 6 (Index: 5):
      Cosine Similarity: 0.3310
      Dot Product: 29.4009
    Document 7 (Index: 6):
      Cosine Similarity: 0.3571
      Dot Product: 25.5130
    Document 8 (Index: 7):
      Cosine Similarity: 0.3169
      Dot Product: 28.3134
    Document 9 (Index: 8):
      Cosine Similarity: 0.3169
      Dot Product: 23.0050
    Document 10 (Index: 9):
      Cosine Similarity: 0.2837
      Dot Product: 22.6540


In [39]:
import numpy as np

def calculate_indicators(embedding_scores):
    all_cosine_scores = []
    all_dot_product_scores = []

    for result in embedding_scores:
        all_cosine_scores.extend(result['cosine_similarity_scores'])
        all_dot_product_scores.extend(result['dot_product_scores'])

    # Calculate statistics for cosine similarity
    mean_cosine = np.mean(all_cosine_scores)
    std_cosine = np.std(all_cosine_scores)
    percentiles_cosine = np.percentile(all_cosine_scores, [5, 25, 50, 75, 95])
    min_cosine = np.min(all_cosine_scores)
    max_cosine = np.max(all_cosine_scores)

    print("Statistics for Cosine Similarity:")
    print(f"  Mean: {mean_cosine:.4f}")
    print(f"  Standard Deviation: {std_cosine:.4f}")
    print(f"  Percentiles (5, 25, 50, 75, 95): {percentiles_cosine}")
    print(f"  Minimum: {min_cosine:.4f}")
    print(f"  Maximum: {max_cosine:.4f}")
    print("-" * 30)

    # Calculate statistics for dot product
    mean_dot_product = np.mean(all_dot_product_scores)
    std_dot_product = np.std(all_dot_product_scores)
    percentiles_dot_product = np.percentile(all_dot_product_scores, [5, 25, 50, 75, 95])
    min_dot_product = np.min(all_dot_product_scores)
    max_dot_product = np.max(all_dot_product_scores)

    print("Statistics for Dot Product:")
    print(f"  Mean: {mean_dot_product:.4f}")
    print(f"  Standard Deviation: {std_dot_product:.4f}")
    print(f"  Percentiles (5, 25, 50, 75, 95): {percentiles_dot_product}")
    print(f"  Minimum: {min_dot_product:.4f}")
    print(f"  Maximum: {max_dot_product:.4f}")

# Assuming 'embedding_scores' is the list of dictionaries you generated
calculate_indicators(embedding_scores)

Statistics for Cosine Similarity:
  Mean: 0.2715
  Standard Deviation: 0.1085
  Percentiles (5, 25, 50, 75, 95): [0.10114941 0.19902414 0.26584566 0.33872352 0.45967103]
  Minimum: -0.1366
  Maximum: 0.9704
------------------------------
Statistics for Dot Product:
  Mean: 20.1752
  Standard Deviation: 10.4474
  Percentiles (5, 25, 50, 75, 95): [ 7.30600696 14.00329304 18.52821732 23.91221714 39.18457851]
  Minimum: -11.0935
  Maximum: 159.5283
